# Session 4 — Gapminder: time & relationships

A new dataset: **Gapminder** — life expectancy, population and GDP per person for 142 countries, every 5 years from 1952 to 2007.
Because it has a time dimension, we can now show **change over time** and **relationships** between numbers.

**You will learn to:**
- explore a new dataset with what you already know
- draw line charts over time
- draw scatter plots with color and size, and use a log scale
- make small multiples and annotate a chart to tell a story

plotly includes the Gapminder dataset, so we import it here just to load the data (we make plotly charts in session 5).

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

sns.set_theme(style="whitegrid")
gm = px.data.gapminder()
gm.head()

## Part 1 — explore, then lines

Use the tools from sessions 1–2 to get to know the data: size, columns, types, which years and countries are included.

In [ ]:
gm.shape

In [ ]:
gm.info()

In [ ]:
print(gm["year"].unique())
print(gm["country"].nunique(), "countries")
print(gm["continent"].value_counts())

In [ ]:
gm.describe().round(1)

In [ ]:
gm07 = gm[gm["year"] == 2007]   # only the latest year — we reuse this table in the charts below
gm07.sort_values("lifeExp", ascending=False).head()

### Line charts = change over time

Each continent has many countries per year. `lineplot` automatically draws the **mean** per year —
it does the `groupby` for you. `errorbar=None` hides the uncertainty band.

In [ ]:
ax = sns.lineplot(data=gm, x="year", y="lifeExp", hue="continent", errorbar=None)
ax.set_title("Average life expectancy by continent")
plt.show()

The same numbers calculated by hand with `groupby` — they match the chart:

In [ ]:
gm.groupby(["year", "continent"])["lifeExp"].mean().unstack().round(1).tail()

For individual countries, filter first with `isin`. `marker="o"` adds a dot at each data point.

In [ ]:
countries = gm[gm["country"].isin(["Iran", "China", "United States"])]
ax = sns.lineplot(data=countries, x="year", y="gdpPercap", hue="country", marker="o")
ax.set_title("GDP per person over time")
ax.set_ylabel("GDP per capita (USD)")
plt.show()

## Part 2 — relationships

A scatter plot shows two numbers per country. You can add a third variable as **color** (`hue`)
and a fourth as **dot size** (`size`). `sizes=(20, 1000)` sets the smallest and largest dot.

Look at the x-axis: most countries are squashed to the left, because a few are extremely rich.

In [ ]:
ax = sns.scatterplot(data=gm07, x="gdpPercap", y="lifeExp", hue="continent",
                     size="pop", sizes=(20, 1000), alpha=0.7)
ax.set_title("Wealth vs. health, 2007")
plt.show()

**Log scale** fixes this: each step on the axis means ×10 instead of +10 (100 → 1,000 → 10,000).
Use it whenever values span a huge range, like money or population. Now the pattern becomes clear.
`move_legend` places the legend outside the chart.

In [ ]:
ax = sns.scatterplot(data=gm07, x="gdpPercap", y="lifeExp", hue="continent",
                     size="pop", sizes=(20, 1000), alpha=0.7)
ax.set_xscale("log")
ax.set_title("Wealth vs. health, 2007 (log scale)")
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.show()

### Small multiples — same chart, one panel per group

`relplot` makes scatter or line charts with one panel per value of `col=`.
Here: one panel per year, so you can compare 1952 → 1982 → 2007 at a glance.

In [ ]:
years = gm[gm["year"].isin([1952, 1982, 2007])]
g = sns.relplot(data=years, x="gdpPercap", y="lifeExp", hue="continent",
                col="year", kind="scatter", height=4)
g.set(xscale="log")
plt.show()

### Polish: annotate the story

A good chart has a title that states the message, clear axis labels, and highlights what matters.
`idxmax()` finds the row with the highest value; `annotate` writes a label with an arrow pointing to it.
`palette` changes the colors.

In [ ]:
best = gm07.loc[gm07["lifeExp"].idxmax()]

fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=gm07, x="gdpPercap", y="lifeExp", hue="continent",
                palette="Set2", ax=ax)
ax.set_xscale("log")
ax.annotate(best["country"], xy=(best["gdpPercap"], best["lifeExp"]),
            xytext=(-60, 10), textcoords="offset points",
            arrowprops={"arrowstyle": "->"})
ax.set_title("Richer countries live longer (2007)")
ax.set_xlabel("GDP per capita (USD, log scale)")
ax.set_ylabel("Life expectancy (years)")
sns.move_legend(ax, "lower right")
plt.show()

## Exercises
Pick **3 countries** you're curious about. In one 1×3 figure show, over time:
1. life expectancy
2. GDP per capita (log scale — for the y-axis use `set_yscale("log")`)
3. population

Title every panel and save as `my_countries.png`.

*Try each one yourself first, then compare with the solution below.*

In [ ]:
# Solution (example countries)
mine = gm[gm["country"].isin(["Iran", "Turkey", "Germany"])]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.lineplot(data=mine, x="year", y="lifeExp", hue="country", ax=axes[0])
axes[0].set_title("Life expectancy")

sns.lineplot(data=mine, x="year", y="gdpPercap", hue="country", ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("GDP per capita (log)")

sns.lineplot(data=mine, x="year", y="pop", hue="country", ax=axes[2])
axes[2].set_title("Population")

fig.tight_layout()
fig.savefig("my_countries.png", dpi=150)
plt.show()

## Summary
- `lineplot` for time, `scatterplot` for relationships; `hue` + `size` add more variables
- log scale for huge ranges
- `relplot(col=...)` for small multiples
- titles, labels and annotations turn a chart into a message

## Practice
1. Bar chart of the 10 richest countries (GDP per capita) in 2007, with horizontal bars (tip: put the country on the y-axis).
2. Average life expectancy per continent: 1952 vs. 2007 side by side.
3. **Write down one question** you want to answer in the mini-project of session 5.

*Try each one yourself first, then compare with the solution below.*

In [ ]:
# Practice solutions
# 1
top10 = gm07.sort_values("gdpPercap", ascending=False).head(10)
ax = sns.barplot(data=top10, x="gdpPercap", y="country")
ax.set_title("10 richest countries, 2007")
plt.show()

# 2
then_now = gm[gm["year"].isin([1952, 2007])]
ax = sns.barplot(data=then_now, x="continent", y="lifeExp", hue="year", errorbar=None)
ax.set_title("Life expectancy 1952 vs 2007")
plt.show()

## LeetCode practice

LeetCode has no chart problems; these practise reshaping tables, which seaborn often needs first. On LeetCode choose **Pandas** as the language. You get a function that receives DataFrames — write your code inside it and `return` the answer as a DataFrame.

2889. [Reshape Data: Pivot](https://leetcode.com/problems/reshape-data-pivot/). Long → wide with `pivot`, like `unstack()`.
1795. [Rearrange Products Table](https://leetcode.com/problems/rearrange-products-table/). Wide → long with `melt` (the shape `hue=` wants); drop the missing rows.
1321. [Restaurant Growth](https://leetcode.com/problems/restaurant-growth/). Change over time: total per day, then a 7-day window with `rolling(7)`.

LeetCode checks your answer, and each problem has an *Editorial* / *Solutions* tab if you get stuck.